In [1]:
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

print("Loading telemetry data...")
# Load your dataset (subsampling if necessary for memory)
df = pd.read_csv('data.csv')

# Split the dataset using the 'period' column
df_train_official = df[df['period'] == 'train'].copy()
df_submission = df[df['period'].isin(['valid', 'test'])].copy()

print(f"Official Train records: {df_train_official.shape[0]}")
print(f"Records to predict (Valid + Test): {df_submission.shape[0]}")

Loading telemetry data...
Official Train records: 34652855
Records to predict (Valid + Test): 29844461


In [2]:
print("Creating Local Split from Official Train data...")

# Extract month and year if you haven't already
df_train_official['timedate'] = pd.to_datetime(df_train_official['timedate'])
df_train_official['year'] = df_train_official['timedate'].dt.year
df_train_official['month'] = df_train_official['timedate'].dt.month

# Define features (drop identifiers, targets, and our new 'period' column)
features = [col for col in df_train_official.columns if col not in ['deviceId', 'timedate', 'x2', 'year', 'month', 'period']]

# Local Train: Oct 2024 - Feb 2025
local_train_mask = (df_train_official['year'] == 2024) | ((df_train_official['year'] == 2025) & (df_train_official['month'] <= 2))
# Local Validation: March 2025 - April 2025
local_val_mask = (df_train_official['year'] == 2025) & (df_train_official['month'].isin([3, 4]))

X_local_train = df_train_official[local_train_mask][features]
y_local_train = df_train_official[local_train_mask]['x2']

X_local_val = df_train_official[local_val_mask][features]
y_local_val = df_train_official[local_val_mask]['x2']

local_val_meta = df_train_official[local_val_mask][['deviceId', 'year', 'month', 'x2']].copy()

Creating Local Split from Official Train data...


In [3]:
print("Training Local LightGBM model...")

local_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

local_model.fit(
    X_local_train, y_local_train,
    eval_set=[(X_local_val, y_local_val)],
    callbacks=[early_stopping(stopping_rounds=50)]
)

print("\nEvaluating Local Model...")
local_val_meta['pred_x2'] = local_model.predict(X_local_val)

monthly_preds = local_val_meta.groupby(['deviceId', 'year', 'month']).agg(
    actual_x2=('x2', 'mean'),
    predicted_x2=('pred_x2', 'mean')
).reset_index()

local_mae = mean_absolute_error(monthly_preds['actual_x2'], monthly_preds['predicted_x2'])
print(f"Local Validation MAE (Monthly Average): {local_mae:.5f}")

Training Local LightGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.238070 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 305
[LightGBM] [Info] Number of data points in the train set: 24647952, number of used features: 15
[LightGBM] [Info] Start training from score 0.178105
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 0.00355602

Evaluating Local Model...
Local Validation MAE (Monthly Average): 0.01385


In [4]:
print("Retraining on ALL official training data...")

# Use the best number of trees found during local validation
best_iterations = local_model.best_iteration_

X_full_train = df_train_official[features]
y_full_train = df_train_official['x2']

final_model = LGBMRegressor(
    n_estimators=best_iterations,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_model.fit(X_full_train, y_full_train)

print("Predicting on official Valid and Test sets...")

# Ensure df_submission has year/month extracted
df_submission['timedate'] = pd.to_datetime(df_submission['timedate'])
df_submission['year'] = df_submission['timedate'].dt.year
df_submission['month'] = df_submission['timedate'].dt.month

# Predict 5-minute intervals for the submission data
df_submission['pred_x2'] = final_model.predict(df_submission[features])

# The submission requires predicting the average x2 for each device and month
print("Aggregating predictions to monthly averages...")
final_submission = df_submission.groupby(['deviceId', 'year', 'month']).agg(
    prediction=('pred_x2', 'mean')
).reset_index()

# Format to exactly match the required columns: deviceld, year, month, prediction
final_submission = final_submission[['deviceId', 'year', 'month', 'prediction']]

# Save to CSV
final_submission.to_csv('submission.csv', index=False)
print("Final submission saved! Ready to upload.")

Retraining on ALL official training data...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.284132 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 309
[LightGBM] [Info] Number of data points in the train set: 34652855, number of used features: 15
[LightGBM] [Info] Start training from score 0.157723
Predicting on official Valid and Test sets...
Aggregating predictions to monthly averages...
Final submission saved! Ready to upload.
